# Sephora E-Commerce EDA: 05 - Business Insights & Value Mapping

This final notebook synthesizes findings from both datasets to deliver business value: pricing indexing, uncovering highly-rated under-exposed products ('hidden gems'), and pointing out overrated luxury items.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_products
from src.viz_utils import set_custom_style

set_custom_style()
products_df = load_products("../data/product_info.csv")

## 1. Value for Money Index

We define a **Value Index** metric:  
$$\text{Value Index} = \frac{\text{Rating} \times \ln(\text{Loves Count} + 1)}{\text{Price (USD)}}$$
This formula rewards high-quality (rating) products with strong social validation (loves count) relative to their price. We filter for items with reviews > 10 and price > 0.

In [ ]:
valid = products_df[(products_df["price_usd"] > 0) & (products_df["reviews"] > 10)].copy()
valid["value_index"] = (valid["rating"] * np.log1p(valid["loves_count"])) / valid["price_usd"]

top_value = valid.sort_values(by="value_index", ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(data=top_value, x="value_index", y="product_name", palette="summer")
plt.title("Top 10 Sephora Value-for-Money Products", fontsize=14, weight="bold")
plt.xlabel("Value for Money Index")
plt.ylabel("Product Name")
plt.show()

## 2. Discovering 'Hidden Gems'

These are products with ratings >= 4.5 and review counts between 10 and 50. They are exceptionally well-rated but haven't received enough reviews to go mainstream. Marketing could benefit from pushing these products.

In [ ]:
hidden_gems = products_df[
    (products_df["rating"] >= 4.5) & 
    (products_df["reviews"] >= 10) & 
    (products_df["reviews"] <= 50)
].sort_values(by="loves_count", ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(data=hidden_gems, x="loves_count", y="product_name", palette="viridis")
plt.title("Top 10 'Hidden Gems' in Sephora Catalog", fontsize=14, weight="bold")
plt.xlabel("Loves Count (Engagement)")
plt.ylabel("Product Name")
plt.show()

## 3. Identifying Overrated Luxury Products

These are items with pricing >= $80, rating <= 3.5, and review count >= 30. Customers are paying premium luxury prices for products with demonstrably below-average ratings, signaling formulation or value problems.

In [ ]:
overrated = products_df[
    (products_df["price_usd"] >= 80) & 
    (products_df["rating"] <= 3.5) & 
    (products_df["reviews"] >= 30)
].sort_values(by="rating", ascending=True).head(10)

if len(overrated) > 0:
    plt.figure(figsize=(12, 6))
    sns.barplot(data=overrated, x="price_usd", y="product_name", palette="autumn")
    plt.title("Luxury Products with Poor Ratings (Price >= $80, Rating <= 3.5)", fontsize=14, weight="bold")
    plt.xlabel("Price (USD)")
    plt.ylabel("Product Name")
    plt.show()
else:
    print("No overrated products found matching threshold.")